# v3 on v4's held-out sets — Kaggle T4

Scores the **v3** checkpoint on the same packed stores, with the same protocol,
that v4 was scored on (`V4_modal/Output/metrics.json`) and that DAV2/V1 is judged
on (`DAV2_V1/README.md` §5, *Scoring*). That puts v3, v4 and DAV2 in one table:

| set | protocol | v4 / DAV2 key |
|---|---|---|
| `gamus/test` (2861 tiles) | centre crop, plain + TTA, all tiles | `test_gamus_test_plain` / `_tta` |
| `gamus/test` | full-tile sliding window + TTA, first 400 | `test_gamus_test_sliding_tta` |
| `india_labeled/val` | centre crop, plain, first 400 | `val_india_labeled` (best epoch) |
| `dfc23_g050/val` | centre crop, plain, first 400 | `val_dfc23_g050` (best epoch) |

TTA is dihedral × scales **(1.0, 1.25)**, v4's and DAV2's `tta_scales`. v3
trained with `(1.0,)`, so `--tta_scales` overrides the run config here. Every
result also carries v4's `per_landscape` breakdown (`eval/landscape.py`, ported
unchanged).

**Why the v3 tree and not v4's `--test_sources`.** v4 kept every parameter name
but changed HeadB's numerics — the pooled feature is RMS-normalised, the width
softmax runs in fp32, the bin widths are floored (`V4_Kaggle/models/heads.py`).
A v3 checkpoint therefore *loads* into v4 without complaint and then predicts
through different bin centres. Wrong numbers, no error. Only the packed store is
shared — `dwdata/packed.py` is byte-identical in both trees, and the val crop
(`TileDataset`) and the sliding window are the same code.

### Before you run
1. **Attach** (Add Input) the packed stores published from the Modal volume —
   the very stores v4 was scored on:
   * `abhaydkale232/depthwizard-gamus` — **required** (`gamus/test`)
   * `abhaydkale232/depthwizard-india-labeled` — `india_labeled/val`
   * a packed `dfc23_g050` store, **if one gets published**. There is none on
     Kaggle yet (only the raw DFC23 Track 2, whose `val` has no DSM), so without
     it v3's DFC23 row is simply absent and the table says so.
2. **Add secrets** `GITHUB_PAT` (the repo is private; read-only
   `contents` scope is enough) and `HF_TOKEN` (DINOv3-SAT is gated — the architecture is built
   from the hub even though the checkpoint then overwrites every weight).
3. Set `HF_REPO` / `HF_FILE` below to wherever `best.pt` lives on the Hub.
4. Optional: attach DAV2/V1's committed output to get its column too.
5. Accelerator: **GPU T4 x2**. `eval_test.py` puts one model copy on each card
   and splits every batch across them, so both work through every stage. Run
   through *Save Version → Save & Run All (Commit)* — an interactive session
   idles out long before this finishes.

Wall clock: the plain pass measured 409 s on a *single* T4 (it reads ~21 GB of
1024 px tiles off the input mount, so a second card helps it least). TTA is
8 dihedral × 2 scales ≈ 20× the plain compute and the 400-tile sliding pass
≈ 11×; two cards roughly halve both. Budget ~2-3 h in total; each stage prints
its own time.

In [ ]:
# ---- what to score, and where it lives -------------------------------
HF_REPO   = "akashch1512/SingleViewHeightEstimator"
HF_FILE   = "best_2.715"       # a file, or a folder holding exactly one .pt
HF_TYPE   = "model"            # "model" or "dataset"

# ---- the protocol: every number below is v4's / DAV2's config.py -----
TTA_SCALES = "1.0,1.25"        # tta_scales (v3 trained with 1.0 only)
TEST_TILES = 0                 # test_tiles: 0 = all 2861 gamus/test tiles
SLIDING    = 400               # test_sliding_tiles
AUX        = ("india_labeled", "dfc23_g050")   # the secondary val sets v4 reported
AUX_TILES  = 400               # val_tiles: first 400 in sorted-stem order
MAX_H      = 150               # max_valid_height_m: v4 ran 150, v3/DAV2 default 200

DAV2_METRICS = ""              # DAV2's metrics.json; "" = look under /kaggle/input

DATA_ROOT = "/kaggle/temp/dwdata"   # symlink farm only, nothing is copied
OUT       = "/kaggle/working/v3_test"
REPO_URL  = "https://github.com/AbhayKale332/DW-Team_ENDRA.git"

assert HF_REPO, "set HF_REPO to the Hub repo holding the v3 best.pt"

## 1 — code + deps

`torch` is left alone: Kaggle's image already carries the CUDA build.


In [ ]:
import os, subprocess, sys, json, time, base64
from pathlib import Path

def sh(cmd, **kw):
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, **kw)

def secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception as e:
        print(f"no Kaggle secret {name}:", e)
        return os.environ.get(name, "")

# The repo is private.  The PAT goes in as a one-off header, never into the
# remote URL: SRC sits under /kaggle/working, so its .git/config is saved with
# the notebook output.  Nor through sh(), which echoes the command.
GITHUB_PAT = secret("GITHUB_PAT")
assert GITHUB_PAT, "GITHUB_PAT is required (the repo is private)"

SRC = Path("/kaggle/working/DepthWizard")
if not SRC.exists():
    auth = base64.b64encode(f"x-access-token:{GITHUB_PAT}".encode()).decode()
    print("$ git clone --depth 1", REPO_URL, SRC, flush=True)
    r = subprocess.run(["git", "-c", f"http.extraHeader=Authorization: Basic {auth}",
                        "clone", "--depth", "1", REPO_URL, str(SRC)],

                       env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
    if r.returncode:  # not check=True: CalledProcessError would print the header
        raise RuntimeError(f"git clone failed (exit {r.returncode}) — "
                           "is GITHUB_PAT valid with read access to the repo?")
V3 = SRC / "FineTunning" / "v3"
assert (V3 / "eval_test.py").is_file(), f"no eval_test.py under {V3}"

sh([sys.executable, "-m", "pip", "install", "-q", "-r", str(V3 / "requirements.txt")])
os.chdir(V3)
print("cwd:", os.getcwd())

In [ ]:
# HF_TOKEN: the encoder is gated, and the checkpoint may be too.
os.environ["HF_TOKEN"] = secret("HF_TOKEN")
assert os.environ["HF_TOKEN"], "HF_TOKEN is required (gated DINOv3-SAT)"
print("HF_TOKEN loaded")

## 2 — the v3 checkpoint

2.5 GB. It is pulled once into `/kaggle/temp` (not `/kaggle/working`, which is
capped at 20 GB and is what the notebook output persists).


In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

# HF_FILE may name a folder (the Hub URL was .../tree/main/best_2.715), so
# resolve it to the one checkpoint inside rather than guessing its name.
if not HF_FILE.endswith(".pt"):
    folder = [f for f in list_repo_files(HF_REPO, repo_type=HF_TYPE,
                                         token=os.environ["HF_TOKEN"])
              if f.startswith(HF_FILE.rstrip("/") + "/")]
    print(f"{HF_REPO}/{HF_FILE}:", folder)
    pts = [f for f in folder if f.endswith(".pt")]
    assert len(pts) == 1, f"expected one .pt under {HF_FILE}, found {pts}"
    HF_FILE = pts[0]

CKPT = hf_hub_download(HF_REPO, HF_FILE, repo_type=HF_TYPE,
                       token=os.environ["HF_TOKEN"],
                       local_dir="/kaggle/temp/ckpt")
print(CKPT, f"{Path(CKPT).stat().st_size / 1e9:.2f} GB")

# The run that produced it — this is what pins decoder_dim / n_bins /
# encoder_feature_indices. eval_test.py refuses to score if the checkpoint
# does not fit the config, rather than half-loading it under strict=False.
RUN_CONFIG = SRC / "Logs" / "v3" / "config.json"
print(json.dumps({k: json.loads(RUN_CONFIG.read_text())[k] for k in
                  ("encoder_model_id", "decoder_dim", "n_bins", "tile_size",
                   "canonical_gsd_m", "tta_scales")}, indent=1))


## 3 — link the packed stores

DAV2's own `run_kaggle.sh link`, so the store → `<source>/<split>` mapping is
the one DAV2 trains and tests through. It finds every `index.json` under
`/kaggle/input` (the datasets mount as
`/kaggle/input/datasets/abhaydkale232/depthwizard-gamus/test/test/`), maps the
slug to the source name (`depthwizard-india-labeled` → `india_labeled`) and
symlinks each split into `DATA_ROOT`. The stores ship `stretch_bounds_2_98.npy`,
so nothing has to be written into the read-only mount.

In [ ]:
sh(["bash", str(SRC / "FineTunning" / "DAV2_V1" / "run_kaggle.sh"), "link"],
   env={**os.environ, "DW_DATA_ROOT": DATA_ROOT})

In [ ]:
def index(name, split):
    f = Path(DATA_ROOT) / name / split / "index.json"
    return json.loads(f.read_text()) if f.is_file() else None

idx = index("gamus", "test")
assert idx, "no gamus/test store — attach abhaydkale232/depthwizard-gamus"
print("gamus/test", {k: v for k, v in idx.items() if k != "shards"})

# The store v4's test numbers came from: 2861 tiles @ 1024 px / 0.33 m.  A
# different count here means this is not the same set of tiles, and the rows of
# the table stop being comparable — say so in the deck rather than quietly
# comparing different problems.
if (idx["n"], idx["tile_px"], idx["gsd_m"]) != (2861, 1024, 0.33):
    print(f"[!] geometry differs from v4's test store (2861, 1024, 0.33) — "
          f"got ({idx['n']}, {idx['tile_px']}, {idx['gsd_m']})")

AUX_OK = []
for s in AUX:
    i = index(s, "val")
    if i:
        AUX_OK.append(s)
        print(f"{s}/val", {k: v for k, v in i.items() if k != "shards"})
    else:
        print(f"[!] {s}/val not attached — skipped; its row will carry v4/DAV2 only")

## 4 — score it

`gamus/test` first, exactly as v4's `--test_sources gamus:test`: every tile
centre-crop plain and TTA, then sliding window + TTA over the first 400. Then
each secondary val set the way v4 scored `val_<source>` every epoch: the first
400 tiles, centre crop, plain, no sliding.

`--amp_dtype fp16` is not a detail: v3 trained under bf16 on an H100, and a T4
is Turing — it has no bf16 units, so leaving the trained value in place gets you
an emulated path that is slow at best. v4's numbers were taken under bf16 on an
H100, so expect fourth-decimal differences from precision alone.

`--batch_size 8 --eval_batch_mult 2` is 16 tiles per forward, split 8 per T4 —
the per-card load that fits a 16 GB T4 with room. It also sets the sliding
window's batch (`batch_size // 2` = 4 windows, i.e. one whole 1024 px tile, 2 per
card). Halve it if you see an OOM.

`--max_valid_height_m 150` is v4's value, not v3's. The cap is part of the valid
mask every metric is computed over, and v3's config (and DAV2's default) say 200 —
left alone, v3 is scored on 150-200 m pixels v4 never saw, and the few huge
errors there move RMSE and tall bias. At 150 the `n px` check below should find
v3 and v4 on identical pixels.

Everything lands in one `OUT/test_metrics.json`: the `test_gamus_test_*` keys
as v4 names them, and `val_<source>` for the secondary sets.

In [ ]:
def score(source, split, *extra):
    out = Path(OUT) / f"{source}_{split}"
    sh([sys.executable, "eval_test.py",
        "--ckpt", CKPT,
        "--run_config", str(RUN_CONFIG),
        "--data_root", DATA_ROOT,
        "--source", source, "--split", split,
        "--tta_scales", TTA_SCALES,
        "--out", str(out),
        "--amp_dtype", "fp16",
        "--max_valid_height_m", str(MAX_H),
        "--batch_size", "8", "--eval_batch_mult", "2",
        "--num_workers", "4", "--prefetch_factor", "4", *extra])
    return json.loads((out / "test_metrics.json").read_text())

t0 = time.time()
m3 = score("gamus", "test", "--tiles", str(TEST_TILES), "--sliding_tiles", str(SLIDING))
for s in AUX_OK:
    # A failure here must not cost the gamus/test numbers above.
    try:
        r = score(s, "val", "--tiles", str(AUX_TILES), "--sliding_tiles", "0",
                  "--skip_tta")
        m3[f"val_{s}"] = r[f"test_{s}_val_plain"]
    except Exception as e:
        print(f"[!] {s}/val failed: {e}")
(Path(OUT) / "test_metrics.json").write_text(json.dumps(m3, indent=2))
print(f"total {(time.time() - t0) / 60:.0f} min -> {Path(OUT) / 'test_metrics.json'}")

## 5 — the table for the deck

v4's column is read out of `V4_modal/Output/metrics.json` in the repo and DAV2's
out of its own `metrics.json`, i.e. the numbers those runs printed — nothing
here is retyped. `test_*` keys are read directly. `val_<source>` is read from
the history record of the epoch with the lowest primary val RMSE, which is the
epoch `best.pt` was saved at.

**Check the `n px` column.** v3 and v4 both centre-crop 512 px through the same
code, so on the same tiles their valid-pixel counts are *identical*; the cell
below says so if they are not. DAV2 crops 518 px, so its centre-crop `n px`
differs by design; its sliding-window count (whole tiles) should still match.

In [ ]:
import pandas as pd

def results(m):
    """{(set, protocol): metrics} out of a metrics.json — v3's, v4's or DAV2's."""
    out = {}
    for suf in ("plain", "tta", "sliding_tta"):
        if m.get(f"test_gamus_test_{suf}"):
            out[("gamus/test", suf)] = m[f"test_gamus_test_{suf}"]
    hist = [h for h in m.get("history") or [] if h.get("val")]
    best = min(hist, key=lambda h: h["val"]["global"]["rmse_m"]) if hist else m
    for s in AUX:
        if best.get(f"val_{s}"):
            out[(f"{s}/val", "plain")] = best[f"val_{s}"]
    return out

m4 = json.loads((SRC / "FineTunning" / "V4_modal" / "Output" / "metrics.json").read_text())
models = {"v3": m3, "v4": m4}
dav2 = ([Path(DAV2_METRICS)] if DAV2_METRICS else
        sorted(Path("/kaggle/input").rglob("dav2-v1/metrics.json"))
        + [SRC / "FineTunning" / "DAV2_V1" / "outputs" / "dav2-v1" / "metrics.json"])
dav2 = [f for f in dav2 if f.is_file()]
if dav2:
    models["DAV2/V1"] = json.loads(dav2[0].read_text())
    print("DAV2/V1 from", dav2[0])
else:
    print("no DAV2/V1 metrics.json found — v3 vs v4 only")

R = {name: results(m) for name, m in models.items()}
ORDER = ([("gamus/test", p) for p in ("plain", "tta", "sliding_tta")]
         + [(f"{s}/val", "plain") for s in AUX])

rows, index = [], []
for key in ORDER:
    for name, r in R.items():
        if key not in r:
            continue
        d, g = r[key], r[key]["global"]
        rows.append({"RMSE (m)": g["rmse_m"], "MAE (m)": g["mae_m"],
                     "Pearson r": g["pearson_r"], "delta1": g["delta1"],
                     "balanced RMSE (m)": d["balanced_rmse_m"],
                     "tall bias (m)": d["tall_gt15m"].get("bias_m"),
                     "flat bias (m)": d["flat_lt1m"].get("bias_m"),
                     "landscape spread (m)": d.get("landscape_rmse_spread_m"),
                     "n px": g["n"]})
        index.append((*key, name))
    missing = [n for n in R if key not in R[n]]
    if missing:
        print(f"[!] {key[0]} {key[1]}: no result for {', '.join(missing)}")
    if key in R["v3"] and key in R["v4"] and \
            R["v3"][key]["global"]["n"] != R["v4"][key]["global"]["n"]:
        print(f"[!] {key[0]} {key[1]}: v3 scored {R['v3'][key]['global']['n']} px, "
              f"v4 {R['v4'][key]['global']['n']} — not the same pixels")

tbl = pd.DataFrame(rows, index=pd.MultiIndex.from_tuples(
    index, names=["set", "protocol", "model"])).round(3)
tbl["n px"] = tbl["n px"].astype("int64")
tbl.to_csv(Path(OUT) / "v3_vs_v4_vs_dav2.csv")
(Path(OUT) / "v3_vs_v4_vs_dav2.md").write_text(tbl.astype(object).to_markdown())
print(tbl.astype(object).to_markdown())
print("\nQuote the gamus/test sliding_tta rows: full-tile sliding window at native"
      " GSD on held-out tiles is what a DSM deliverable actually is.")
tbl

In [ ]:
# Per-stratum and per-landscape, for the slide that explains where the error
# lives.  Same gamus/test protocol for every model.
key = ("gamus/test", "sliding_tta" if ("gamus/test", "sliding_tta") in R["v3"] else "tta")
cols = {}
for name, r in R.items():
    if key not in r:
        continue
    ps = {s: d for s, d in r[key]["per_stratum"].items() if d.get("n")}
    cols[f"{name} RMSE (m)"] = {s: d["rmse_m"] for s, d in ps.items()}
    cols[f"{name} bias (m)"] = {s: d["bias_m"] for s, d in ps.items()}
g3 = R["v3"][key]
cols["px frac"] = {s: d["n"] / g3["global"]["n"]
                   for s, d in g3["per_stratum"].items() if d.get("n")}
st = pd.DataFrame(cols).round(3)
st.to_csv(Path(OUT) / "per_stratum.csv")
print(key, "\n" + st.to_markdown())

land = pd.DataFrame({
    f"{name} RMSE (m)": {k: d["rmse_m"] for k, d in r[key].get("per_landscape", {}).items()}
    for name, r in R.items() if key in r}).round(3)
land["tiles"] = pd.Series({k: d["tiles"] for k, d in g3.get("per_landscape", {}).items()})
land.to_csv(Path(OUT) / "per_landscape.csv")
print("\n" + land.to_markdown())
st

## 6 — the test report

v4's `viz/report_html.py`, pointed at v3's numbers. `build()` reads
`<dir>/metrics.json` and already prefers `test_*` keys over val ones, so
`test_metrics.json` plus v3's run config is all it needs. There are no training
figures here, so the report shows only the tables and headline cards. The output is
one self-contained HTML file in `OUT`.

In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "report_html", SRC / "FineTunning" / "V4_Kaggle" / "viz" / "report_html.py")
report_html = importlib.util.module_from_spec(spec)
spec.loader.exec_module(report_html)

rep_dir = Path(OUT) / "report"
rep_dir.mkdir(parents=True, exist_ok=True)
m = json.loads((Path(OUT) / "test_metrics.json").read_text())
m.setdefault("config", json.loads(RUN_CONFIG.read_text()))
(rep_dir / "metrics.json").write_text(json.dumps(m, indent=2))

rep = report_html.build(rep_dir, out_path=Path(OUT) / "test_report.html",
                        title="DepthWizard v3")
# build() hard-codes "validation report" and a v4 footer; this is v3 on test.
rep.write_text(rep.read_text(encoding="utf-8")
               .replace("validation report", "test report")
               .replace("DepthWizard v4 ·", "DepthWizard v3 ·"), encoding="utf-8")
print(rep)